**Shows the Process of HCS and LCS Samples for the SVC Models**

In [ ]:
import pandas as pd
import numpy as np
import regex as re
import nltk
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

import textwrap
import string

from nltk.corpus import wordnet
from nltk.tag import pos_tag

import swifter
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report

In [46]:
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet
from nltk.tokenize import word_tokenize

nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('stopwords')

[nltk_data] Downloading package wordnet to /Users/emma/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /Users/emma/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt to /Users/emma/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/emma/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package stopwords to /Users/emma/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [120]:
labeled = pd.read_csv('/Users/emma/Desktop/labeled_data_C.csv')
labeled = labeled[labeled['my_label'] != 'unclear'].reset_index(drop=True)
labeled['my_label'].value_counts()

my_label
no stance    1738
prochoice     985
prolife       759
Name: count, dtype: int64

In [121]:
labeled['preprocessed'] = (
    labeled['body']
    .str.replace(f"[{string.punctuation}]", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
)


In [ ]:
lemmatizer = WordNetLemmatizer()

def get_wordnet_pos(word):

    tag = pos_tag([word])[0][1][0].upper()
    tag_dict = {"J": wordnet.ADJ, "N": wordnet.NOUN, "V": wordnet.VERB, "R": wordnet.ADV}
    return tag_dict.get(tag, wordnet.NOUN)

def lemmatize_text(text):
    if isinstance(text, str):
        words = word_tokenize(text)
        lemmatized_words = [lemmatizer.lemmatize(word, get_wordnet_pos(word)) for word in words]
        return " ".join(lemmatized_words)
    return text

In [123]:
labeled['preprocessed'] = labeled['body'].swifter.apply(lemmatize_text)

Pandas Apply:   0%|          | 0/3482 [00:00<?, ?it/s]

In [124]:
# encoding labels
label_encoder = LabelEncoder()
y  = label_encoder.fit_transform(labeled['my_label'])
(dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))))

{'no stance': 0, 'prochoice': 1, 'prolife': 2}

In [ ]:
eng = sorted(ENGLISH_STOP_WORDS)
eng_string = ', '.join(eng)

textwrap.fill(eng_string, width=80)
  


'a, about, above, across, after, afterwards, again, against, all, almost, alone,\nalong, already, also, although, always, am, among, amongst, amoungst, amount,\nan, and, another, any, anyhow, anyone, anything, anyway, anywhere, are, around,\nas, at, back, be, became, because, become, becomes, becoming, been, before,\nbeforehand, behind, being, below, beside, besides, between, beyond, bill, both,\nbottom, but, by, call, can, cannot, cant, co, con, could, couldnt, cry, de,\ndescribe, detail, do, done, down, due, during, each, eg, eight, either, eleven,\nelse, elsewhere, empty, enough, etc, even, ever, every, everyone, everything,\neverywhere, except, few, fifteen, fifty, fill, find, fire, first, five, for,\nformer, formerly, forty, found, four, from, front, full, further, get, give, go,\nhad, has, hasnt, have, he, hence, her, here, hereafter, hereby, herein,\nhereupon, hers, herself, him, himself, his, how, however, hundred, i, ie, if,\nin, inc, indeed, interest, into, is, it, its, itsel

In [ ]:
tfidf = TfidfVectorizer(max_features = 1500,
                        ngram_range = (1,2),
                        stop_words='english'
                        )

X  = tfidf.fit_transform(labeled['preprocessed'])

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, stratify=y, random_state=13)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=13)

svm_model = SVC(kernel="linear", 
                C= .4, 
                probability=True, 
                 class_weight={0: 1.0, 1: 3.0, 2: 2.0})

svm_model.fit(X_train, y_train)

y_train_pred = svm_model.predict(X_train)
y_val_pred = svm_model.predict(X_val)

print("Train Accuracy:", accuracy_score(y_train, y_train_pred))
print('Validation Accuracy:', accuracy_score(y_val, y_val_pred))
print(classification_report(y_val, y_val_pred, target_names=label_encoder.classes_))
print("")

scores = cross_val_score(svm_model, X_train, y_train, cv=5, scoring='accuracy')
print("CV Scores:", scores)
print("CV Std Dev:", np.std(scores))

print(confusion_matrix(y_val, y_val_pred))


Train Accuracy: 0.8785391875256463
Validation Accuracy: 0.8007662835249042
              precision    recall  f1-score   support

   no stance       0.90      0.76      0.83       260
   prochoice       0.69      0.81      0.74       148
     prolife       0.79      0.88      0.83       114

    accuracy                           0.80       522
   macro avg       0.79      0.82      0.80       522
weighted avg       0.81      0.80      0.80       522


CV Scores: [0.79508197 0.79098361 0.81519507 0.78850103 0.81519507]
CV Std Dev: 0.011786223651897205
[[198  46  16]
 [ 17 120  11]
 [  5   9 100]]


In [138]:
y_test_pred = svm_model.predict(X_test)
print("Final Test Accuracy:", accuracy_score(y_test, y_test_pred))


Final Test Accuracy: 0.8068833652007649


In [80]:
X_train_indices, X_temp_indices, y_train, y_temp = train_test_split(
    labeled.index, y, test_size=0.3, stratify=y, random_state=13
)

X_val_indices, X_test_indices, y_val, y_test = train_test_split(
    X_temp_indices, y_temp, test_size=0.5, stratify=y_temp, random_state=13
)

def get_missclassified(y_test, y_pred, X_test_indices, n):
    y_test = np.array(y_test)
    y_pred = np.array(y_pred)

    misclassified = np.where(y_test != y_pred)[0]

    for i in misclassified[:n]:
        original_index = X_test_indices[i]
        print(f"True Label: {label_encoder.classes_[y_test[i]]}")
        print(f"Predicted Label: {label_encoder.classes_[y_pred[i]]}")
        text = labeled.loc[original_index, 'body']
        wrapped_text = textwrap.fill(text, width=80)
        print(wrapped_text)
        print('--' * 50)
        print("")

get_missclassified(y_test, y_test_pred, X_test_indices, 100)

True Label: no stance
Predicted Label: prolife
the babies of the babies of the babies getting abortions too
----------------------------------------------------------------------------------------------------

True Label: no stance
Predicted Label: prochoice
you seem to want to have a debate about the merits of abortion. good luck with
that.
----------------------------------------------------------------------------------------------------

True Label: prolife
Predicted Label: no stance
this is why we should seek out prolife doctors. speak with your money. i feel
unsafe receiving treatment from one of these.
----------------------------------------------------------------------------------------------------

True Label: no stance
Predicted Label: prolife
and this was what killed that women in ireland
----------------------------------------------------------------------------------------------------

True Label: no stance
Predicted Label: prochoice
quit projecting how youd react if so

In [111]:
# getting confident preds

import numpy as np

proba = svm_model.predict_proba(X_test)

threshold = .8

y_pred = np.argmax(proba, axis =1)

max_proba = proba.max(axis=1)

conf_pred = max_proba > threshold

uncertain = ~conf_pred

y_pred[uncertain] = -1

In [ ]:
# getting unconfident preds

# import numpy as np

# proba = svm_model.predict_proba(X_test)

# threshold = .5

# y_pred = np.argmax(proba, axis =1)

# max_proba = proba.max(axis=1)

# unconf_pred = max_proba < threshold

# conf = ~unconf_pred

# y_pred[conf] = -1

In [ ]:
# adding more confident labels
# unlabeled = pd.read_parquet('/Users/emma/Desktop/thesis/actual_folder/clean/total_comments_B.parquet').sample(5000)

# unlabeled = unlabeled[~unlabeled['id'].isin(labeled['id'])]

# unlabeled.shape

(4990, 9)

In [ ]:
# unlabeled['preprocessed'] = (
#     unlabeled['body']
#     .str.replace(f"[{string.punctuation}]", "", regex=True)
#     .str.replace(r"\s+", " ", regex=True)
# )
# unlabeled['preprocessed'] = unlabeled['preprocessed'].swifter.apply(lemmatize_text)

Pandas Apply:   0%|          | 0/4990 [00:00<?, ?it/s]

In [114]:
X_ul = tfidf.transform(unlabeled['preprocessed'])
proba_ul = svm_model.predict_proba(X_ul)

threshold = .7

y_pred_ul = np.argmax(proba_ul, axis = 1)

max_proba_ul = proba_ul.max(axis = 1)

conf_pred_ul = max_proba_ul > threshold

uncertain_pred_ul = ~conf_pred_ul

y_pred_ul[uncertain_pred_ul] = -1


In [ ]:
# unlabeled['predicted_label'] = y_pred_ul

In [ ]:
unlabeled['predicted_label'].value_counts()

# unlabeled[unlabeled['predicited_label'] != -1].to_csv('uncertain.csv', index=False)

predicted_label
 0    3754
-1    1030
 1     162
 2      44
Name: count, dtype: int64

In [117]:
unlabeled[unlabeled['predicted_label'] == 2].to_csv('pl_conf.csv', index=False)

In [ ]:
unlabeled[unlabeled['predicited_label'] == 1].to_csv('pc_conf.csv', index=False)

In [95]:
unlabeled[unlabeled['predicited_label'] == 0].sample(100).to_csv('ns_conf.csv', index=False)